In [5]:
import numpy as np
import pandas as pd
import glob
import musclebeachtools as mbt
import samb_work.dimtools as dimt
import os

from natsort import natsorted
from pathlib import Path
import joblib
from datetime import datetime
from matplotlib.widgets import SpanSelector, Button
from matplotlib.colors import PowerNorm
from PIL import Image
from tqdm import tqdm
import tkinter as tk
from tkinter import simpledialog
from IPython.display import display, clear_output
import ipywidgets as widgets

import sahara_work as saw
import musclebeachtools as mbt
import neuraltoolkit as ntk
import matplotlib.pyplot as plt

# from james_work import crit_utils as cu
from sahara_work import crit_utils as cu
%load_ext autoreload
%autoreload 2
import os
NCPU = "4"
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '4'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family':'DejaVu Sans'})
#import swifter

/hlabhome/wg-mjames/.conda/envs/james_test_copy/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/hlabhome/wg-mjames/.conda/envs/james_test_copy/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


In [21]:
def get_100_isi_timings(isis):
    output = []
    if len(isis):
        #print(isis[0], isis[-1])
        for time_ind in range(0,len(isis)-1, 100):
            output.append(isis[time_ind])
        output.append(isis[-1])
    return output

In [22]:
def get_sleeptimes(sleepnpy):
    #deal with sleepybois
    sleep_states = np.zeros((2,0))
    sleeps = np.load(sleepnpy)
    timestamps = (np.nonzero(np.diff(sleeps))[0]+1)*4
    time_ind = (timestamps/4)-1
    states = sleeps[time_ind.astype(int)]
    s = np.array(states)
    t = np.stack((timestamps,s))
    sleep_states = np.concatenate((sleep_states,t), axis =1)
    start_time = np.append([0],sleep_states[0,:-1])
    stop_time = sleep_states[0,:]
    sleepstate = sleep_states[1,:]
    sleepdf = pd.DataFrame({'start_time': start_time,'stop_time': stop_time,'sleepstate': sleepstate})
    return(sleepdf)

# make into a function
def get_isi_df(cell_dir, sleepnpy_dir, animal_name, fr_cutoff=50):
    animal_name = animal_name.lower()
    cells = np.load(cell_dir, allow_pickle=True)
    # good_cells = [cell for cell in cells if cell.quality < 3 and cell.plotFR(binsz=cell.end_time, lplot=0, lonoff=0)[0][0] < fr_cutoff and cell.presence_ratio() > .99]

    # rstart time is the same
    start_time = cells[0].rstart_time
    start_time = datetime.strptime(start_time, '%Y-%m-%d_%H-%M-%S')
    # cu = james_work crit_utils
    birthday = cu.get_birthday(animal_name)
    age = start_time - birthday
    # convert age to decimal days
    age = age.days + age.seconds/(24*60*60)
    geno = cu.get_genotype(animal_name)
    sex = cu.get_sex(animal_name)
    sleep_df = get_sleeptimes(sleepnpy_dir)
    print('got initial info')
    # cell_dir = '/media/HlabShare/Clustering_Data/ZBR00114/12hr_bins/zbr114_04262024/0_12/probe2/co/H_2024-04-26_17-25-37_2024-04-27_05-20-38_neurons_group0.npy'
    # sleepnpy_dir = '/media/HlabShare/Ben_work/data/ZBR114/0426/0_12/ZBR114_0426_0_12_sleep.npy'
    record_time = sleepnpy_dir.split('/')
    epoch = record_time[-3] + '_' + record_time[-2]
    # display(sleep_df)
    # brain_region = [cu.get_regions(animal_name)]

    # cell_type and peak_latency are not the same for all cells
    # cell_type = cells[0].cell_type
    # peak_latency = cells[0].peaklatency


    # loop through each cell of good_cells
    # get the spike times for each cell
    spike_time_and_state = pd.DataFrame()
    
    for neuron_index, neuron in enumerate(cells):
        neuron_spike_times = neuron.spike_time_sec
        for index, row in sleep_df.iterrows():
            start_time = row['start_time']
            stop_time = row['stop_time']
            sleep_state = row['sleepstate']
            cell_spikes = neuron_spike_times[(neuron_spike_times >= start_time) & (neuron_spike_times <= stop_time)]

            spike_time_and_state = pd.concat([spike_time_and_state, pd.DataFrame({'cell_no': neuron_index,
                                                                                  'spikes': [cell_spikes], 
                                                                                  'sleep_state': sleep_state, 
                                                                                  'cell_type': neuron.cell_type, 
                                                                                  'peak_latency': neuron.peaklatency, 
                                                                                  'region': neuron.region,
                                                                                  'length': int(stop_time-start_time)})])

    # change 4, 5 to 1 first
    # merging mircoarousal and quiet wake into wake
    #spike_time_and_state['sleep_state'] = spike_time_and_state['sleep_state'].replace(4, 1)
    spike_time_and_state['sleep_state'] = spike_time_and_state['sleep_state'].replace(5, 1)

    #display(spike_time_and_state)

    # separate the dataframes by cell number
    cells = spike_time_and_state['cell_no'].unique()

    # display(cells)

    cell_isi = pd.DataFrame()

    for cell_index, cell in enumerate(cells):
        cell_df = spike_time_and_state[spike_time_and_state['cell_no'] == cell_index]

#         print(cell_index)
#         print(len(cell_df))

        cell_type = cell_df['cell_type'].iloc[0]
        peak_latency = cell_df['peak_latency'].iloc[0]
        brain_region = cell_df['region'].iloc[0]

        wake = cell_df[cell_df['sleep_state'] == 1]
        nrem = cell_df[cell_df['sleep_state'] == 2]
        rem = cell_df[cell_df['sleep_state'] == 3]
        ma = cell_df[cell_df['sleep_state']==4]

        wake_isi = []
        nrem_isi = []
        rem_isi = []
        ma_isi = []
        wake_isi_timings, nrem_isi_timings, rem_isi_timings = [],[],[]
        wake_lens, nrem_lens, rem_lens, ma_lens = [],[],[], []
        for index, row in wake.iterrows():
            spikes = row['spikes']
            isi = np.diff(spikes)
            wake_isi.append(isi)
            wake_isi_timings.append(get_100_isi_timings(spikes))
            wake_lens.append(row['length'])
        for index, row in nrem.iterrows():
            spikes = row['spikes']
            isi = np.diff(spikes)
            nrem_isi.append(isi)
            nrem_isi_timings.append(get_100_isi_timings(spikes))
            nrem_lens.append(row['length'])
        for index, row in rem.iterrows():
            spikes = row['spikes']
            isi = np.diff(spikes)
            rem_isi.append(isi)
            rem_isi_timings.append(get_100_isi_timings(spikes))
            rem_lens.append(row['length'])
#         for index, row in ma.iterrows():
#             spikes = row['spikes']
#             isi = np.diff(spikes)
#             ma_isi.append(isi)
#             ma_lens.append(row['length'])

        cell_isi = pd.concat([cell_isi, pd.DataFrame({
                                                            'animal': animal_name,
                                                            'epoch': epoch,
                                                            'age':age,
                                                            'sex': sex,
                                                            'geno': geno,
                                                            'brain_region': brain_region,
                                                            'cell_type': cell_type,
                                                            'peak_latency': peak_latency,
                                                            'cell_no': cell_index,
                                                            'wake_isi': [wake_isi],
                                                            'nrem_isi': [nrem_isi],
                                                            'rem_isi': [rem_isi],
                                                            #'ma_isi': [ma_isi],
                                                            'wake_lens': [wake_lens],
                                                            'nrem_lens': [nrem_lens],
                                                            'rem_lens': [rem_lens],
                                                            'wake_timings': [wake_isi_timings],
                                                            'nrem_timings': [nrem_isi_timings],
                                                            'rem_timings': [rem_isi_timings],
                                                            #'ma_lens': [ma_lens]
                                                            })])

    # reset index to cell_no
    cell_isi = cell_isi.reset_index(drop=True)
    #display(cell_isi)
    return cell_isi

# function to get cell_dir and sleepnpy_dir based on animal name
def get_dirs(animal_name, cell_base_dir, sleepnpy_base_dir):
    character_code = animal_name[:3]
    character_code = character_code.upper()
    number_code = animal_name[-3:]
    # add zeros in front of number_code to make the length of 5
    number_code = number_code.zfill(5)
    # '/media/HlabShare/appps1_cortical_communication_samb/appps1/data/ZBR117/0529/0_12/ZBR117_0529_0_12_singleunits.npy'
    cell_dir = natsorted(glob.glob(f'{cell_base_dir}{animal_name.lower()}/*/*_*/*_singleunits.npy'))
    sleepnpy_dir = natsorted(glob.glob(f'{sleepnpy_base_dir}{animal_name.upper()}/*/*_*/*_sleep.npy'))
    return cell_dir, sleepnpy_dir

animal_names = [
    'caf69',
    'caf71',
    'caf73',
    'caf74',
    'caf75',
    'caf77',
    'caf81',
    'caf82',
    'zbr106',
    'zbr113',
    'zbr114',
    'zbr117',
    'zbr118',
    'zbr119',
    'bth90',
    'bth95',
    'bth100'
    ]
print(len(animal_names))

cell_base_dir = '/media/HlabShare/appps1_cortical_communication_samb/appps1/data/'
sleepnpy_base_dir = '/media/HlabShare/appps1_cortical_communication_samb/appps1/data/'
all_animal_isi_df = pd.DataFrame()
for animal_name in animal_names:
    print('Working on', animal_name)
    cell_dir, sleepnpy_dir = get_dirs(animal_name, cell_base_dir, sleepnpy_base_dir)
    print(f"Has {len(cell_dir)} files to go through")
    for i in range(len(cell_dir)):
        isi_df = get_isi_df(cell_dir[i], sleepnpy_dir[i], animal_name)
        display(isi_df)
        all_animal_isi_df = pd.concat([all_animal_isi_df, isi_df], ignore_index=True)
display(all_animal_isi_df)

17
Working on caf69
Has 3 files to go through
got initial info


,animal,epoch,age,sex,geno,brain_region,cell_type,peak_latency,cell_no,wake_isi,nrem_isi,rem_isi,wake_lens,nrem_lens,rem_lens,wake_timings,nrem_timings,rem_timings
0,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.68,0,"[[0.00592, 0.00132, 0.004360000000000001, 0.02...","[[0.15272000000004482, 0.07171999999991385, 0....","[[0.15104000000019369, 0.11567999999988388, 0....","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.00532, 8.30984, 16.46344, 25.3846, 37.0028...","[[3320.05296, 3328.55344, 3339.40532, 3345.584...","[[3841.05772, 3868.16824, 3891.42572, 3909.411..."
1,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.68,1,"[[0.12224, 6.87172, 1.7081599999999995, 2.1617...","[[14.897280000000137, 1.6376399999999194, 6.17...","[[3.8209999999999127, 4.546280000000024, 3.097...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[4e-05, 219.53304, 509.02504, 573.5778, 737.8...","[[3329.31148, 3785.66544], [4008.16452, 4438.9...","[[3859.45592, 3980.94744], [4470.14316, 4478.7..."
2,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.52,2,"[[0.06476000000000001, 0.017119999999999996, 0...","[[0.05519999999978609, 0.8142000000002554, 0.0...","[[0.05375999999978376, 0.286200000000008, 0.08...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.01684, 22.50024, 46.03396, 66.695, 89.9888...","[[3320.1932, 3372.29616, 3432.87716, 3497.7676...","[[3840.76872, 3894.55876, 3944.32072, 3989.872..."
3,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.52,3,"[[0.24439999999999998, 0.27224000000000004, 0....","[[2.0253600000000915, 0.87920000000031, 0.7080...","[[0.6291599999999562, 0.11724000000003798, 0.7...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.00892, 22.6154, 48.60976, 73.74956, 91.015...","[[3320.42208, 3368.56088, 3431.02976, 3503.225...","[[3840.58268, 3899.72988, 3942.88864, 3976.861..."
4,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.48,4,"[[0.15028000000000002, 0.34468000000000004, 0....","[[0.59996000000001, 0.4675600000000486, 0.8568...","[[2.2804799999998977, 0.46628000000009706, 1.0...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.25464, 28.47056, 53.57136, 85.74888, 91.68...","[[3320.71772, 3373.96056, 3425.57584, 3523.176...","[[3841.0414, 3907.79736, 3946.2996, 3986.7532,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,caf69,1228_0_12,358.152546,F,wt,CA1,FS,0.16,108,"[[34.46592, 6.195040000000006, 4.6371599999999...","[[2.4536400000001777, 126.5286799999999], [53....","[[], [], [], [], [], [], [0.8602799999998751],...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[32.80516, 3241.5028], [4615.8082, 4616.21448...","[[3585.84156, 3714.82388], [4004.17008, 4179.7...","[[], [4462.15176], [], [], [], [], [6402.35072..."
109,caf69,1228_0_12,358.152546,F,wt,CA1,FS,0.16,109,"[[0.011119999999999991, 0.25624, 1.09736000000...","[[0.00864000000001397, 0.017479999999977736, 0...","[[0.6227999999996428, 0.01224000000001979, 0.0...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.16508, 22.5774, 66.36284, 107.02832, 139.5...","[[3321.97504, 3369.055, 3399.23044, 3427.95528...","[[3840.5284, 3865.65348, 3884.4952, 3893.80336..."
110,caf69,1228_0_12,358.152546,F,wt,CA1,FS,0.36,110,"[[34.998279999999994, 0.0036400000000043065, 9...","[[196.11527999999998], [], [],

got initial info


,animal,epoch,age,sex,geno,brain_region,cell_type,peak_latency,cell_no,wake_isi,nrem_isi,rem_isi,wake_lens,nrem_lens,rem_lens,wake_timings,nrem_timings,rem_timings
0,caf69,1230_0_12,360.350069,F,wt,ACC,RSU,0.68,0,"[[0.008239999999999997, 0.12388000000000002, 0...","[[0.15499999999997272, 0.026360000000067885, 0...","[[0.1572000000001026, 0.2180799999998726, 0.20...","[464, 32, 4, 28, 132, 12, 48, 32, 76, 80, 4, 6...","[440, 800, 380, 240, 60, 540, 500, 400, 560, 1...","[80, 120, 100, 120, 120, 40, 40, 120, 160, 36,...","[[0.06716, 6.26484, 10.93276, 16.27832, 21.464...","[[1400.03316, 1409.95224, 1419.47252, 1431.124...","[[1840.10032, 1850.59672, 1862.92868, 1868.039..."
1,caf69,1230_0_12,360.350069,F,wt,ACC,RSU,0.64,1,"[[0.03216, 0.024640000000000002, 0.00299999999...","[[0.010199999999940701, 0.03920000000016444, 0...","[[0.007920000000012806, 0.0768399999999474, 0....","[464, 32, 4, 28, 132, 12, 48, 32, 76, 80, 4, 6...","[440, 800, 380, 240, 60, 540, 500, 400, 560, 1...","[80, 120, 100, 120, 120, 40, 40, 120, 160, 36,...","[[4e-05, 3.97644, 9.08664, 13.49404, 18.98232,...","[[1400.1804, 1406.86436, 1415.66032, 1421.9640...","[[1840.07752, 1845.03056, 1854.02816, 1861.824..."
2,caf69,1230_0_12,360.350069,F,wt,ACC,RSU,0.60,2,"[[0.07152, 0.04124, 0.34524, 0.486200000000000...","[[0.03435999999987871, 0.024040000000013606, 0...","[[0.4557200000001558, 0.01756000000000313, 0.1...","[464, 32, 4, 28, 132, 12, 48, 32, 76, 80, 4, 6...","[440, 800, 380, 240, 60, 540, 500, 400, 560, 1...","[80, 120, 100, 120, 120, 40, 40, 120, 160, 36,...","[[0.00068, 19.50456, 36.82864, 82.88096, 127.8...","[[1400.07176, 1441.48528, 1497.01032, 1555.473...","[[1840.08784, 1868.06584, 1892.7318, 1919.4996..."
3,caf69,1230_0_12,360.350069,F,wt,ACC,RSU,0.64,3,"[[0.003239999999999993, 0.11176, 0.30811999999...","[[0.008000000000038199, 0.2400399999999081, 0....","[[0.08047999999985223, 0.06144000000017513, 0....","[464, 32, 4, 28, 132, 12, 48, 32, 76, 80, 4, 6...","[440, 800, 380, 240, 60, 540, 500, 400, 560, 1...","[80, 120, 100, 120, 120, 40, 40, 120, 160, 36,...","[[0.07208, 15.09396, 40.64872, 155.25012, 193....","[[1400.2988, 1627.93472, 1837.92856], [1920.56...","[[1840.04204, 1891.45236, 1919.93704], [2721.2..."
4,caf69,1230_0_12,360.350069,F,wt,ACC,RSU,0.64,4,"[[0.0022000000000000075, 0.021079999999999988,...","[[0.1311599999999089, 0.007160000000112632, 0....","[[0.14148000000000138, 0.13011999999980617, 0....","[464, 32, 4, 28, 132, 12, 48, 32, 76, 80, 4, 6...","[440, 800, 380, 240, 60, 540, 500, 400, 560, 1...","[80, 120, 100, 120, 120, 40, 40, 120, 160, 36,...","[[0.06712, 13.9256, 26.83008, 86.28036, 152.70...","[[1400.29888, 1490.87144, 1656.444, 1839.95796...","[[1840.18404, 1882.1058, 1915.54608], [2720.55..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,caf69,1230_0_12,360.350069,F,wt,CA1,FS,0.24,98,"[[0.08600000000000001, 0.007719999999999991, 0...","[[0.038840000000163855, 0.03971999999998843, 0...","[[0.024840000000040163, 0.015079999999898064, ...","[464, 32, 4, 28, 132, 12, 48, 32, 76, 80, 4, 6...","[440, 800, 380, 240, 60, 540, 500, 400, 560, 1...","[80, 120, 100, 120, 120, 40, 40, 120, 160, 36,...","[[4e-05, 7.55952, 13.87992, 21.56788, 29.29604...","[[1400.00696, 1407.1888, 1411.57212, 1419.9694...","[[1840.03044, 1844.78644, 1851.2142, 1857.9278..."
99,caf69,1230_0_12,360.350069,F,wt,CA1,FS,0.24,99,"[[0.13440000000000002, 0.41267999999999994, 0....","[[0.5940000000000509, 0.04048000000011598, 1.1...","[[0.07395999999994274, 0.048800000000028376, 0...","[464, 32, 4, 28, 132, 12, 48, 32, 76, 80, 4, 6...","[440, 800, 380, 240, 60, 540, 500, 400, 560, 1...","[80, 120, 100, 120, 120, 40, 40, 120, 160, 36,...","[[0.05336, 4.73884, 18.24964, 25.60804, 27.609...","[[1400.4026, 1423.42572, 1440.73824, 1460.3150...","[[1840.0506, 1855.24728, 1871.2616, 1880.62536..."
100,caf69,1230_0_12,360.350069,F,wt,CA1,RSU,0.84,100,"[[0.06183999999999999, 0.42504, 0.214960000000...","[[0.19707999999991443, 0.10860000

got initial info


...

got initial info


,animal,epoch,age,sex,geno,brain_region,cell_type,peak_latency,cell_no,wake_isi,nrem_isi,rem_isi,wake_lens,nrem_lens,rem_lens,wake_timings,nrem_timings,rem_timings
0,bth100,0624_0_12,98.187037,M,wt,CA1,RSU,0.52,0,"[[0.49512, 1.2418400000000003, 0.1206399999999...","[[0.21515999999996893, 0.026039999999966312, 0...","[[1.3489199999999073, 1.434479999999894, 0.657...","[400, 1344, 56, 256, 2328, 396, 1088, 256, 632...","[748, 496, 208, 704, 608, 132, 1072, 1072, 444...","[48, 60, 52, 220, 24, 20, 84, 52, 20, 28, 20, ...","[[0.31228, 62.92228, 145.27932, 190.00316, 236...","[[1996.833, 2078.86012, 2167.73916, 2328.71804...","[[2744.05456, 2791.91], [3351.82384, 3403.7572..."
1,bth100,0624_0_12,98.187037,M,wt,CA1,RSU,0.64,1,"[[0.4216800000000003, 0.3255600000000003, 0.56...","[[1.979280000000017, 1.270240000000058, 0.9627...","[[21.82448000000022, 0.17671999999993204, 4.17...","[400, 1344, 56, 256, 2328, 396, 1088, 256, 632...","[748, 496, 208, 704, 608, 132, 1072, 1072, 444...","[48, 60, 52, 220, 24, 20, 84, 52, 20, 28, 20, ...","[[6.32168, 236.44084, 291.98144, 372.262, 399....","[[1999.97976, 2164.5922, 2321.87324, 2524.2451...","[[2749.07176, 2791.97052], [3346.48112, 3373.3..."
2,bth100,0624_0_12,98.187037,M,wt,CA1,FS,0.24,2,"[[0.2242, 0.40735999999999994, 0.94112, 0.5941...","[[0.10572000000001935, 0.04927999999995336, 0....","[[0.2140799999997398, 0.026920000000245636, 0....","[400, 1344, 56, 256, 2328, 396, 1088, 256, 632...","[748, 496, 208, 704, 608, 132, 1072, 1072, 444...","[48, 60, 52, 220, 24, 20, 84, 52, 20, 28, 20, ...","[[0.31512, 22.27644, 42.65252, 64.48636, 81.29...","[[1996.0532, 2001.56284, 2004.58892, 2008.2158...","[[2744.11052, 2752.7358, 2760.5368, 2772.47724..."
3,bth100,0624_0_12,98.187037,M,wt,CA1,RSU,0.64,3,"[[0.08784, 0.34644, 0.04496, 0.252999999999999...","[[0.0065600000000358705, 0.25499999999988177, ...","[[0.4331600000000435, 0.6678399999996145, 0.35...","[400, 1344, 56, 256, 2328, 396, 1088, 256, 632...","[748, 496, 208, 704, 608, 132, 1072, 1072, 444...","[48, 60, 52, 220, 24, 20, 84, 52, 20, 28, 20, ...","[[0.01684, 20.32564, 39.8282, 56.45168, 75.578...","[[1996.0806, 2030.57468, 2064.78504, 2099.9717...","[[2744.10388, 2774.47096, 2789.85208], [3344.1..."
4,bth100,0624_0_12,98.187037,M,wt,CA1,RSU,0.48,4,"[[0.10651999999999999, 0.10172000000000003, 0....","[[0.321920000000091, 0.6134799999999814, 0.625...","[[4.153760000000148, 10.17452000000003, 0.2336...","[400, 1344, 56, 256, 2328, 396, 1088, 256, 632...","[748, 496, 208, 704, 608, 132, 1072, 1072, 444...","[48, 60, 52, 220, 24, 20, 84, 52, 20, 28, 20, ...","[[0.07692, 63.1494, 157.99788, 255.81748, 372....","[[1996.69844, 2098.37524, 2196.75204, 2352.937...","[[2744.48404, 2790.86588], [3345.24848, 3402.7..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,bth100,0624_0_12,98.187037,M,wt,V1,FS,0.28,93,"[[0.024599999999999997, 0.021599999999999994, ...","[[0.2716400000001613, 0.0051199999998061685, 0...","[[0.007920000000012806, 0.08611999999993714, 0...","[400, 1344, 56, 256, 2328, 396, 1088, 256, 632...","[748, 496, 208, 704, 608, 132, 1072, 1072, 444...","[48, 60, 52, 220, 24, 20, 84, 52, 20, 28, 20, ...","[[0.0704, 7.91516, 14.69508, 18.90348, 22.9378...","[[1996.23112, 2002.61064, 2025.44752, 2036.865...","[[2744.3654, 2755.83204, 2763.99208, 2776.0762..."
94,bth100,0624_0_12,98.187037,M,wt,V1,FS,0.28,94,"[[0.08967999999999998, 0.14088000000000003, 0....","[[0.6038800000001174, 0.9746399999999085, 0.63...","[[0.23471999999992477, 0.1178000000004431, 0.0...","[400, 1344, 56, 256, 2328, 396, 1088, 256, 632...","[748, 496, 208, 704, 608, 132, 1072, 1072, 444...","[48, 60, 52, 220, 24, 20, 84, 52, 20, 28, 20, ...","[[0.038, 16.24452, 28.43676, 58.4462, 71.3236,...","[[1996.94916, 2093.61924, 2168.96412, 2247.810...","[[2744.16288, 2762.37084, 2778.47244, 2791.997..."
95,bth100,0624_0_12,98.187037,M,wt,V1,FS,0.24,95,"[[0.007200000000000001, 0.02096, 0.02308000000...","[[0.0027199999999538704, 0.0074400000000

got initial info


,animal,epoch,age,sex,geno,brain_region,cell_type,peak_latency,cell_no,wake_isi,nrem_isi,rem_isi,wake_lens,nrem_lens,rem_lens,wake_timings,nrem_timings,rem_timings
0,bth100,0624_12_24,98.68706,M,wt,CA1,FS,0.12,0,"[[0.025560000000041327, 0.013760000000047512, ...","[[0.0024400000000000047, 0.0023199999999999957...","[[0.012600000000020373, 0.0058399999999210195,...","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.02812, 670.26996, 671.89888, 673.5458, 6...","[[0.04176, 2.55352, 6.58892, 9.05928, 12.22428...","[[592.00368, 595.76548, 600.50304, 603.74572, ..."
1,bth100,0624_12_24,98.68706,M,wt,CA1,FS,0.12,1,"[[0.11644000000001142, 0.03663999999992029, 0....","[[0.16071999999999997, 0.00804000000000002, 0....","[[0.007000000000061846, 0.035520000000019536, ...","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.0832, 679.6716, 684.66272, 687.21084, 69...","[[0.0364, 15.34164, 22.97888, 33.68256, 44.257...","[[592.13852, 598.03596, 605.5396, 611.5196, 61..."
2,bth100,0624_12_24,98.68706,M,wt,CA1,FS,0.12,2,"[[0.12408000000004904, 0.5085599999999886, 0.5...","[[6.20472, 4.413679999999999, 1.21828, 2.12264...","[[0.18988000000001648, 1.0691600000000108, 1.3...","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.18252, 765.64232, 874.85624, 982.93896, ...","[[0.64512, 107.75448, 195.08272, 345.67664, 46...","[[592.3836, 667.98692], [7225.85152, 7239.5913..."
3,bth100,0624_12_24,98.68706,M,wt,CA1,RSU,0.68,3,"[[1.084920000000011, 6.257839999999987, 7.3813...","[[2.22504, 2.5909199999999997, 0.0122800000000...","[[0.6865199999999732, 5.437440000000038, 0.467...","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[674.59012, 923.73884, 984.71692, 1006.88036,...","[[1.84264, 227.45684, 416.10672, 571.7464, 591...","[[594.61712, 654.12812], [7217.08904, 7220.399..."
4,bth100,0624_12_24,98.68706,M,wt,CA1,FS,0.24,4,"[[0.01984000000004471, 0.008800000000064756, 0...","[[0.02068, 0.013920000000000002, 0.01911999999...","[[0.010480000000029577, 0.02255999999999858, 0...","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.03672, 669.97724, 671.94212, 674.09624, ...","[[0.0028, 1.87252, 3.63784, 5.68228, 7.51496, ...","[[592.0144, 593.792, 595.84156, 598.016, 599.4..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,bth100,0624_12_24,98.68706,M,wt,V1,RSU,0.68,90,"[[0.020479999999906795, 0.4212400000000116, 0....","[[1.57676, 0.04683999999999999, 0.407960000000...","[[1.6818399999999656, 0.047159999999962565, 0....","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.57176, 707.01432, 740.97008, 767.3638, 7...","[[1.30304, 45.60452, 92.60128, 153.071, 202.79...","[[592.2716, 636.20772, 667.88588], [7214.34452..."
91,bth100,0624_12_24,98.68706,M,wt,V1,RSU,0.56,91,"[[0.1151199999999335, 0.12744000000009237, 0.0...","[[0.46224, 1.79184, 0.24311999999999978, 0.452...","[[1.3544399999999541, 1.002520000000004, 0.383...","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.24004, 694.74624, 733.93376, 754.5682, 7...","[[0.79468, 44.809, 98.434, 153.49844, 192.7348...","[[592.14768, 617.384, 640.14308, 656.65148, 66..."
92,bth100,0624_12_24,98.68706,M,wt,V1,RSU,0.52,92,"[[0.2658400000000256, 0.22303999999996904, 0.1...","[[0.04723999999999995, 0.015519999999999

,animal,epoch,age,sex,geno,brain_region,cell_type,peak_latency,cell_no,wake_isi,nrem_isi,rem_isi,wake_lens,nrem_lens,rem_lens,wake_timings,nrem_timings,rem_timings
0,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.68,0,"[[0.00592, 0.00132, 0.004360000000000001, 0.02...","[[0.15272000000004482, 0.07171999999991385, 0....","[[0.15104000000019369, 0.11567999999988388, 0....","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.00532, 8.30984, 16.46344, 25.3846, 37.0028...","[[3320.05296, 3328.55344, 3339.40532, 3345.584...","[[3841.05772, 3868.16824, 3891.42572, 3909.411..."
1,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.68,1,"[[0.12224, 6.87172, 1.7081599999999995, 2.1617...","[[14.897280000000137, 1.6376399999999194, 6.17...","[[3.8209999999999127, 4.546280000000024, 3.097...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[4e-05, 219.53304, 509.02504, 573.5778, 737.8...","[[3329.31148, 3785.66544], [4008.16452, 4438.9...","[[3859.45592, 3980.94744], [4470.14316, 4478.7..."
2,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.52,2,"[[0.06476000000000001, 0.017119999999999996, 0...","[[0.05519999999978609, 0.8142000000002554, 0.0...","[[0.05375999999978376, 0.286200000000008, 0.08...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.01684, 22.50024, 46.03396, 66.695, 89.9888...","[[3320.1932, 3372.29616, 3432.87716, 3497.7676...","[[3840.76872, 3894.55876, 3944.32072, 3989.872..."
3,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.52,3,"[[0.24439999999999998, 0.27224000000000004, 0....","[[2.0253600000000915, 0.87920000000031, 0.7080...","[[0.6291599999999562, 0.11724000000003798, 0.7...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.00892, 22.6154, 48.60976, 73.74956, 91.015...","[[3320.42208, 3368.56088, 3431.02976, 3503.225...","[[3840.58268, 3899.72988, 3942.88864, 3976.861..."
4,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.48,4,"[[0.15028000000000002, 0.34468000000000004, 0....","[[0.59996000000001, 0.4675600000000486, 0.8568...","[[2.2804799999998977, 0.46628000000009706, 1.0...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.25464, 28.47056, 53.57136, 85.74888, 91.68...","[[3320.71772, 3373.96056, 3425.57584, 3523.176...","[[3841.0414, 3907.79736, 3946.2996, 3986.7532,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5470,bth100,0624_12_24,98.687060,M,wt,V1,RSU,0.68,90,"[[0.020479999999906795, 0.4212400000000116, 0....","[[1.57676, 0.04683999999999999, 0.407960000000...","[[1.6818399999999656, 0.047159999999962565, 0....","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.57176, 707.01432, 740.97008, 767.3638, 7...","[[1.30304, 45.60452, 92.60128, 153.071, 202.79...","[[592.2716, 636.20772, 667.88588], [7214.34452..."
5471,bth100,0624_12_24,98.687060,M,wt,V1,RSU,0.56,91,"[[0.1151199999999335, 0.12744000000009237, 0.0...","[[0.46224, 1.79184, 0.24311999999999978, 0.452...","[[1.3544399999999541, 1.002520000000004, 0.383...","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.24004, 694.74624, 733.93376, 754.5682, 7...","[[0.79468, 44.809, 98.434, 153.49844, 192.7348...","[[592.14768, 617.384, 640.14308, 656.65148, 66..."
5472,bth100,0624_12_24,98.687060,M,wt,V1,RSU,0.52,92,"[[0.2658400000000256, 0.22303999999996904, 0.1...","[[0.04723999999999995, 0.01

In [23]:
all_animal_isi_df

,animal,epoch,age,sex,geno,brain_region,cell_type,peak_latency,cell_no,wake_isi,nrem_isi,rem_isi,wake_lens,nrem_lens,rem_lens,wake_timings,nrem_timings,rem_timings
0,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.68,0,"[[0.00592, 0.00132, 0.004360000000000001, 0.02...","[[0.15272000000004482, 0.07171999999991385, 0....","[[0.15104000000019369, 0.11567999999988388, 0....","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.00532, 8.30984, 16.46344, 25.3846, 37.0028...","[[3320.05296, 3328.55344, 3339.40532, 3345.584...","[[3841.05772, 3868.16824, 3891.42572, 3909.411..."
1,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.68,1,"[[0.12224, 6.87172, 1.7081599999999995, 2.1617...","[[14.897280000000137, 1.6376399999999194, 6.17...","[[3.8209999999999127, 4.546280000000024, 3.097...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[4e-05, 219.53304, 509.02504, 573.5778, 737.8...","[[3329.31148, 3785.66544], [4008.16452, 4438.9...","[[3859.45592, 3980.94744], [4470.14316, 4478.7..."
2,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.52,2,"[[0.06476000000000001, 0.017119999999999996, 0...","[[0.05519999999978609, 0.8142000000002554, 0.0...","[[0.05375999999978376, 0.286200000000008, 0.08...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.01684, 22.50024, 46.03396, 66.695, 89.9888...","[[3320.1932, 3372.29616, 3432.87716, 3497.7676...","[[3840.76872, 3894.55876, 3944.32072, 3989.872..."
3,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.52,3,"[[0.24439999999999998, 0.27224000000000004, 0....","[[2.0253600000000915, 0.87920000000031, 0.7080...","[[0.6291599999999562, 0.11724000000003798, 0.7...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.00892, 22.6154, 48.60976, 73.74956, 91.015...","[[3320.42208, 3368.56088, 3431.02976, 3503.225...","[[3840.58268, 3899.72988, 3942.88864, 3976.861..."
4,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.48,4,"[[0.15028000000000002, 0.34468000000000004, 0....","[[0.59996000000001, 0.4675600000000486, 0.8568...","[[2.2804799999998977, 0.46628000000009706, 1.0...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.25464, 28.47056, 53.57136, 85.74888, 91.68...","[[3320.71772, 3373.96056, 3425.57584, 3523.176...","[[3841.0414, 3907.79736, 3946.2996, 3986.7532,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5470,bth100,0624_12_24,98.687060,M,wt,V1,RSU,0.68,90,"[[0.020479999999906795, 0.4212400000000116, 0....","[[1.57676, 0.04683999999999999, 0.407960000000...","[[1.6818399999999656, 0.047159999999962565, 0....","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.57176, 707.01432, 740.97008, 767.3638, 7...","[[1.30304, 45.60452, 92.60128, 153.071, 202.79...","[[592.2716, 636.20772, 667.88588], [7214.34452..."
5471,bth100,0624_12_24,98.687060,M,wt,V1,RSU,0.56,91,"[[0.1151199999999335, 0.12744000000009237, 0.0...","[[0.46224, 1.79184, 0.24311999999999978, 0.452...","[[1.3544399999999541, 1.002520000000004, 0.383...","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.24004, 694.74624, 733.93376, 754.5682, 7...","[[0.79468, 44.809, 98.434, 153.49844, 192.7348...","[[592.14768, 617.384, 640.14308, 656.65148, 66..."
5472,bth100,0624_12_24,98.687060,M,wt,V1,RSU,0.52,92,"[[0.2658400000000256, 0.22303999999996904, 0.1...","[[0.04723999999999995, 0.01

In [24]:
import pickle
begin = "/media/HlabShare/Toby_work/sleep_indiv_neuron/"
with open(f'{begin}full_animal_isi.pkl', 'wb') as file:
    pickle.dump(all_animal_isi_df, file)

In [27]:
filtered_dataset = all_animal_isi_df
valid_days = {'caf69':'', 'caf71':'0103','caf73':'0120','caf74':'0125', 'caf75':'0125',
             'caf77':'0202','caf81':'0304','caf82':'0304',
             'zbr106':'0223','zbr113':'0513','zbr114':'0426','zbr117':'','zbr118':'0522',
             'zbr119':'0604','bth90':'0323','bth95':'0611','bth100':'0624'}
for animal in valid_days:
    for file in filtered_dataset[filtered_dataset['animal']==animal]['epoch'].unique():
        if valid_days[animal] not in file:
            filtered_dataset = filtered_dataset[filtered_dataset['epoch']!=file]
filtered_dataset = filtered_dataset[filtered_dataset['animal'].isin(valid_days.keys())]
display(filtered_dataset)

,animal,epoch,age,sex,geno,brain_region,cell_type,peak_latency,cell_no,wake_isi,nrem_isi,rem_isi,wake_lens,nrem_lens,rem_lens,wake_timings,nrem_timings,rem_timings
0,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.68,0,"[[0.00592, 0.00132, 0.004360000000000001, 0.02...","[[0.15272000000004482, 0.07171999999991385, 0....","[[0.15104000000019369, 0.11567999999988388, 0....","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.00532, 8.30984, 16.46344, 25.3846, 37.0028...","[[3320.05296, 3328.55344, 3339.40532, 3345.584...","[[3841.05772, 3868.16824, 3891.42572, 3909.411..."
1,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.68,1,"[[0.12224, 6.87172, 1.7081599999999995, 2.1617...","[[14.897280000000137, 1.6376399999999194, 6.17...","[[3.8209999999999127, 4.546280000000024, 3.097...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[4e-05, 219.53304, 509.02504, 573.5778, 737.8...","[[3329.31148, 3785.66544], [4008.16452, 4438.9...","[[3859.45592, 3980.94744], [4470.14316, 4478.7..."
2,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.52,2,"[[0.06476000000000001, 0.017119999999999996, 0...","[[0.05519999999978609, 0.8142000000002554, 0.0...","[[0.05375999999978376, 0.286200000000008, 0.08...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.01684, 22.50024, 46.03396, 66.695, 89.9888...","[[3320.1932, 3372.29616, 3432.87716, 3497.7676...","[[3840.76872, 3894.55876, 3944.32072, 3989.872..."
3,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.52,3,"[[0.24439999999999998, 0.27224000000000004, 0....","[[2.0253600000000915, 0.87920000000031, 0.7080...","[[0.6291599999999562, 0.11724000000003798, 0.7...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.00892, 22.6154, 48.60976, 73.74956, 91.015...","[[3320.42208, 3368.56088, 3431.02976, 3503.225...","[[3840.58268, 3899.72988, 3942.88864, 3976.861..."
4,caf69,1228_0_12,358.152546,F,wt,ACC,RSU,0.48,4,"[[0.15028000000000002, 0.34468000000000004, 0....","[[0.59996000000001, 0.4675600000000486, 0.8568...","[[2.2804799999998977, 0.46628000000009706, 1.0...","[3320, 320, 20, 1708, 588, 92, 740, 252, 68, 3...","[520, 440, 40, 240, 180, 200, 180, 220, 180, 6...","[160, 40, 100, 20, 40, 60, 100, 40, 108, 80, 3...","[[0.25464, 28.47056, 53.57136, 85.74888, 91.68...","[[3320.71772, 3373.96056, 3425.57584, 3523.176...","[[3841.0414, 3907.79736, 3946.2996, 3986.7532,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5470,bth100,0624_12_24,98.687060,M,wt,V1,RSU,0.68,90,"[[0.020479999999906795, 0.4212400000000116, 0....","[[1.57676, 0.04683999999999999, 0.407960000000...","[[1.6818399999999656, 0.047159999999962565, 0....","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.57176, 707.01432, 740.97008, 767.3638, 7...","[[1.30304, 45.60452, 92.60128, 153.071, 202.79...","[[592.2716, 636.20772, 667.88588], [7214.34452..."
5471,bth100,0624_12_24,98.687060,M,wt,V1,RSU,0.56,91,"[[0.1151199999999335, 0.12744000000009237, 0.0...","[[0.46224, 1.79184, 0.24311999999999978, 0.452...","[[1.3544399999999541, 1.002520000000004, 0.383...","[5852, 336, 168, 2548, 676, 3524, 3780, 448, 3...","[592, 692, 68, 484, 324, 196, 380, 472, 256, 5...","[76, 28, 40, 88, 40, 72, 120, 104, 100, 88, 44...","[[668.24004, 694.74624, 733.93376, 754.5682, 7...","[[0.79468, 44.809, 98.434, 153.49844, 192.7348...","[[592.14768, 617.384, 640.14308, 656.65148, 66..."
5472,bth100,0624_12_24,98.687060,M,wt,V1,RSU,0.52,92,"[[0.2658400000000256, 0.22303999999996904, 0.1...","[[0.04723999999999995, 0.01

In [28]:
with open(f'{begin}filtered_full_animal_isi.pkl', 'wb') as file:
    pickle.dump(filtered_dataset, file)